In [1]:
# import os, itertools
# import numpy as np, pandas as pd
# from sklearn.metrics import roc_auc_score
# from scipy.stats import rankdata

# COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
# LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
#           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
#           "Contusion", "Fracture"]

# def find_file(name):
#     for root, dirs, files in os.walk("/kaggle/input"):
#         dirs[:] = [d for d in dirs if d != "competitions"]
#         if name in files:
#             return os.path.join(root, name)
#     raise FileNotFoundError(name)

# train = pd.read_csv(f"{COMP}/train.csv")
# gold = train[train[LABELS].notna().all(axis=1)][["StudyInstanceUID"] + LABELS].reset_index(drop=True)
# Y = gold[LABELS].values.astype(int)

# FILES = {"effb0": "gold_preds.csv",
#          "resnet50": "gold_preds_resnet50.csv",
#          "dinov2s": "gold_preds_dinov2s.csv"}
# P = {}
# for name, fn in FILES.items():
#     path = find_file(fn)
#     print(f"{name:9s} -> {path}")
#     df = gold[["StudyInstanceUID"]].merge(pd.read_csv(path), on="StudyInstanceUID", how="left")
#     assert df[LABELS].notna().all().all(), f"{name}: ada studi gold tanpa prediksi"
#     P[name] = df[LABELS].values
# print("Studi gold:", len(gold))

# def per_class_auc(y, p):
#     return np.array([roc_auc_score(y[:, k], p[:, k]) if len(np.unique(y[:, k])) == 2 else np.nan
#                      for k in range(len(LABELS))])

# def rank_norm(p):
#     """Ubah probabilitas jadi peringkat 0-1 per kelas (AUC hanya peduli urutan)."""
#     return np.column_stack([rankdata(p[:, k]) / len(p) for k in range(p.shape[1])])

# def make_ens(combo, method):
#     ps = [rank_norm(P[n]) if method == "rank" else P[n] for n in combo]
#     return np.mean(ps, axis=0)

In [2]:
# tab = pd.DataFrame({name: per_class_auc(Y, p) for name, p in P.items()}, index=LABELS)
# tab["terbaik"] = tab[list(P)].idxmax(axis=1)
# tab.loc["MEAN", list(P)] = tab[list(P)].mean()
# print(tab.round(3).to_string())

In [3]:
# rows = []
# names = list(P)
# for r in range(1, len(names) + 1):
#     for combo in itertools.combinations(names, r):
#         for method in (["-"] if r == 1 else ["mean", "rank"]):
#             ens = make_ens(combo, method)
#             rows.append({"models": " + ".join(combo), "method": method, "n": r,
#                          "mean_auc": np.nanmean(per_class_auc(Y, ens))})
# res = pd.DataFrame(rows).sort_values("mean_auc", ascending=False).reset_index(drop=True)
# print(res.round(4).to_string(index=False))

In [4]:
# # 58 studi itu sedikit; bootstrap memperkirakan apakah selisih vs EffB0 bisa kebetulan.
# cands = res[res["n"] > 1].reset_index(drop=True)
# ens_list = {f"{r.models} ({r.method})": make_ens(r.models.split(" + "), r.method)
#             for r in cands.itertuples()}

# rng = np.random.default_rng(0)
# B = 500
# diffs = {k: [] for k in ens_list}
# for _ in range(B):
#     idx = rng.integers(0, len(Y), len(Y))
#     base = np.nanmean(per_class_auc(Y[idx], P["effb0"][idx]))
#     for k, e in ens_list.items():
#         diffs[k].append(np.nanmean(per_class_auc(Y[idx], e[idx])) - base)

# boot = pd.DataFrame([{"ensemble": k,
#                       "selisih_vs_effb0": np.mean(v),
#                       "CI95_bawah": np.percentile(v, 2.5),
#                       "CI95_atas": np.percentile(v, 97.5),
#                       "P(lebih baik)": np.mean(np.array(v) > 0)} for k, v in diffs.items()])
# print(boot.sort_values("selisih_vs_effb0", ascending=False).round(4).to_string(index=False))

In [5]:
# best = cands.iloc[0]
# ens = make_ens(best["models"].split(" + "), best["method"])
# cmp = pd.DataFrame({"effb0": per_class_auc(Y, P["effb0"]),
#                     "ensemble": per_class_auc(Y, ens)}, index=LABELS)
# cmp["selisih"] = cmp["ensemble"] - cmp["effb0"]
# cmp.loc["MEAN"] = cmp.mean()
# print(f"Ensemble: {best['models']} ({best['method']})")
# print(cmp.round(3).to_string())

In [6]:
# for tag in ["dinov2s", "resnet50"]:
#     print(f"\n===== {tag} =====")
#     print(pd.read_csv(find_file(f"train_log_{tag}.csv")).round(4).to_string(index=False))

In [7]:
# import os, itertools
# import numpy as np, pandas as pd
# from sklearn.metrics import roc_auc_score

# COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
# LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
#           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
#           "Contusion", "Fracture"]

# def find_file(name):
#     for root, dirs, files in os.walk("/kaggle/input"):
#         dirs[:] = [d for d in dirs if d != "competitions"]
#         if name in files:
#             return os.path.join(root, name)
#     raise FileNotFoundError(name)

# def auc_per_class(y, p):
#     """AUC per kelas; label kosong dan 0.5 (uncertain) diabaikan."""
#     out = []
#     for k in range(len(LABELS)):
#         ok = ~np.isnan(y[:, k]) & (y[:, k] != 0.5)
#         yk = y[ok, k]
#         out.append(roc_auc_score(yk, p[ok, k]) if len(np.unique(yk)) == 2 else np.nan)
#     return np.array(out)

# # ---- Gold (58 studi, label emas) ----
# train = pd.read_csv(f"{COMP}/train.csv")
# gold = train[train[LABELS].notna().all(axis=1)][["StudyInstanceUID"] + LABELS].reset_index(drop=True)
# Yg = gold[LABELS].values.astype(float)
# GOLD_FILES = {"effb0": "gold_preds.csv",
#               "effb0_s2": "gold_preds_effb0_s2.csv",
#               "resnet50": "gold_preds_resnet50.csv",
#               "dinov2s_v2": "gold_preds_dinov2s_v2.csv"}
# G = {}
# for name, fn in GOLD_FILES.items():
#     df = gold[["StudyInstanceUID"]].merge(pd.read_csv(find_file(fn)), on="StudyInstanceUID", how="left")
#     assert df[LABELS].notna().all().all(), f"{name}: ada studi gold tanpa prediksi"
#     G[name] = df[LABELS].values

# # ---- Validasi (435 studi, label laporan; split sama untuk semua model) ----
# rep = pd.read_csv(find_file("report_labels.csv"))
# VAL_FILES = {"effb0_s2": "val_preds_effb0_s2.csv",
#              "resnet50": "val_preds_resnet50.csv",
#              "dinov2s_v2": "val_preds_dinov2s_v2.csv"}
# V, base = {}, None
# for name, fn in VAL_FILES.items():
#     df = pd.read_csv(find_file(fn))
#     if base is None:
#         base = df[["StudyInstanceUID"]]
#     V[name] = base.merge(df, on="StudyInstanceUID", how="left")[LABELS].values
# Yv = base.merge(rep, on="StudyInstanceUID", how="left")[LABELS].values.astype(float)

# print(f"Gold: {len(Yg)} studi | Val: {len(Yv)} studi")

In [8]:
# tv = pd.DataFrame({n: auc_per_class(Yv, p) for n, p in V.items()}, index=LABELS)
# tg = pd.DataFrame({n: auc_per_class(Yg, p) for n, p in G.items()}, index=LABELS)
# tv.loc["MEAN"] = tv.mean(); tg.loc["MEAN"] = tg.mean()
# print("===== VAL (435 studi) =====");  print(tv.round(3).to_string())
# print("\n===== GOLD (58 studi) =====");  print(tg.round(3).to_string())

In [9]:
# rows = []
# names = list(V)          # model yang punya prediksi val
# for r in range(1, len(names) + 1):
#     for combo in itertools.combinations(names, r):
#         pv = np.mean([V[n] for n in combo], axis=0)
#         rows.append({"models": " + ".join(combo), "n": r,
#                      "val_auc": np.nanmean(auc_per_class(Yv, pv))})
# res = pd.DataFrame(rows).sort_values("val_auc", ascending=False).reset_index(drop=True)

# # Konfirmasi di gold; varian "+ effb0" menambahkan EffB0 seed 42 (model LB 0.842)
# def gold_auc(models):
#     return np.nanmean(auc_per_class(Yg, np.mean([G[n] for n in models], axis=0)))
# res["gold_auc"] = [gold_auc(m.split(" + ")) for m in res["models"]]
# res["gold_auc + effb0"] = [gold_auc(m.split(" + ") + ["effb0"]) for m in res["models"]]
# print(res.round(4).to_string(index=False))

In [10]:
# top = res.iloc[0]["models"].split(" + ")
# cands = {"ensemble terbaik (val)": top,
#          "ensemble terbaik + effb0": top + ["effb0"],
#          "semua 4 model": list(G)}
# rng = np.random.default_rng(0)
# B = 1000
# diffs = {k: [] for k in cands}
# for _ in range(B):
#     idx = rng.integers(0, len(Yg), len(Yg))
#     base_auc = np.nanmean(auc_per_class(Yg[idx], G["effb0"][idx]))
#     for k, models in cands.items():
#         p = np.mean([G[n] for n in models], axis=0)
#         diffs[k].append(np.nanmean(auc_per_class(Yg[idx], p[idx])) - base_auc)

# print("Pembanding: EffB0 seed 42 (submission LB 0.842)\n")
# for k, v in diffs.items():
#     v = np.array(v)
#     print(f"{k:28s} {' + '.join(cands[k]):45s} selisih {v.mean():+.4f} "
#           f"CI95 [{np.percentile(v, 2.5):+.4f}, {np.percentile(v, 97.5):+.4f}] "
#           f"P(lebih baik) {np.mean(v > 0):.0%}")

In [11]:
# import os, glob, numpy as np, pandas as pd
# from sklearn.metrics import roc_auc_score
# import matplotlib.pyplot as plt

# COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
# LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
#           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
#           "Contusion", "Fracture"]

# def find_file(name):
#     for root, dirs, files in os.walk("/kaggle/input"):
#         dirs[:] = [d for d in dirs if d != "competitions"]
#         if name in files:
#             return os.path.join(root, name)
#     raise FileNotFoundError(name)

# def auc_per_class(y, p):
#     out = []
#     for k in range(len(LABELS)):
#         ok = ~np.isnan(y[:, k]) & (y[:, k] != 0.5)
#         yk = y[ok, k]
#         out.append(roc_auc_score(yk, p[ok, k]) if len(np.unique(yk)) == 2 else np.nan)
#     return np.array(out)

# # Label emas, label laporan, dan prediksi guru (rata-rata 5 model fold) untuk 58 studi gold
# train = pd.read_csv(f"{COMP}/train.csv")
# gold = train[train[LABELS].notna().all(axis=1)][["StudyInstanceUID"] + LABELS].reset_index(drop=True)
# Yg = gold[LABELS].values.astype(float)

# rep = pd.read_csv(find_file("report_labels.csv"))
# R = gold[["StudyInstanceUID"]].merge(rep, on="StudyInstanceUID", how="left")[LABELS].values.astype(float)

# fold_dir = os.path.dirname(find_file("gold_preds_effb0_kf_f0.csv"))
# Tg = np.mean([gold[["StudyInstanceUID"]].merge(pd.read_csv(f), on="StudyInstanceUID")[LABELS].values
#               for f in sorted(glob.glob(f"{fold_dir}/gold_preds_effb0_kf_f*.csv"))], axis=0)

# # Kualitas label campuran terhadap label emas, untuk berbagai alpha
# rows = []
# for a in np.round(np.arange(1.0, -0.01, -0.1), 1):
#     mix = np.where(np.isnan(R), Tg, a * R + (1 - a) * Tg)   # label laporan kosong -> pakai guru
#     per = auc_per_class(Yg, mix)
#     rows.append({"alpha": a, "mean_auc": np.nanmean(per), **dict(zip(LABELS, per))})
# res = pd.DataFrame(rows)
# print(res[["alpha", "mean_auc"]].round(4).to_string(index=False))

# best = res.loc[res["mean_auc"].idxmax()]
# print(f"\nAlpha terbaik: {best['alpha']}  (mean AUC {best['mean_auc']:.4f})")
# print("\nPer kelas: label laporan saja (alpha=1) vs campuran terbaik vs guru saja (alpha=0)")
# cmp = res.set_index("alpha").loc[[1.0, best["alpha"], 0.0], LABELS].T
# cmp.columns = ["laporan", f"campuran a={best['alpha']}", "guru"]
# print(cmp.round(3).to_string())

# plt.figure(figsize=(6, 4))
# plt.plot(res["alpha"], res["mean_auc"], marker="o")
# plt.xlabel("alpha (porsi label laporan)"); plt.ylabel("Mean AUC vs label emas")
# plt.title("Kualitas label campuran"); plt.gca().invert_xaxis(); plt.grid(alpha=0.3)
# plt.tight_layout(); plt.show()

In [12]:
# import os, glob, numpy as np, pandas as pd
# from sklearn.metrics import roc_auc_score

# COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
# LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
#           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
#           "Contusion", "Fracture"]

# def find_file(name):
#     for root, dirs, files in os.walk("/kaggle/input"):
#         dirs[:] = [d for d in dirs if d != "competitions"]
#         if name in files:
#             return os.path.join(root, name)
#     raise FileNotFoundError(name)

# def auc_mean(y, p):
#     out = []
#     for k in range(len(LABELS)):
#         ok = ~np.isnan(y[:, k]) & (y[:, k] != 0.5)
#         yk = y[ok, k]
#         out.append(roc_auc_score(yk, p[ok, k]) if len(np.unique(yk)) == 2 else np.nan)
#     return np.nanmean(out)

# train = pd.read_csv(f"{COMP}/train.csv")
# gold = train[train[LABELS].notna().all(axis=1)][["StudyInstanceUID"] + LABELS].reset_index(drop=True)
# Yg = gold[LABELS].values.astype(float)
# load_g = lambda fn: gold[["StudyInstanceUID"]].merge(pd.read_csv(find_file(fn)), on="StudyInstanceUID")[LABELS].values

# G = {"effb0": load_g("gold_preds.csv"),
#      "effb0_s2": load_g("gold_preds_effb0_s2.csv"),
#      "dinov2s_v2": load_g("gold_preds_dinov2s_v2.csv"),
#      "student_v1": load_g("gold_preds_effb0_student_v1.csv"),
#      "student_v2": load_g("gold_preds_effb0_student_v2.csv"),
#      "kf5": np.mean([load_g(f"gold_preds_effb0_kf_f{i}.csv") for i in range(5)], axis=0)}

# A = ["effb0", "effb0_s2", "dinov2s_v2"]                      # ensemble LB 0.871
# CANDS = {"A (LB 0.871)": A,
#          "A + student_v2": A + ["student_v2"],
#          "A + student_v1 + student_v2": A + ["student_v1", "student_v2"],
#          "A + kf5": A + ["kf5"],
#          "A + kf5 + student_v2": A + ["kf5", "student_v2"],
#          "dinov2s_v2 + student_v2": ["dinov2s_v2", "student_v2"]}

# ens = {k: np.mean([G[m] for m in v], axis=0) for k, v in CANDS.items()}
# rng = np.random.default_rng(0)
# idxs = [rng.integers(0, len(Yg), len(Yg)) for _ in range(1000)]
# base_boot = np.array([auc_mean(Yg[i], ens["A (LB 0.871)"][i]) for i in idxs])

# rows = []
# for k, p in ens.items():
#     boot = np.array([auc_mean(Yg[i], p[i]) for i in idxs]) - base_boot
#     rows.append({"kandidat": k, "n_model": len(CANDS[k]), "gold_auc": auc_mean(Yg, p),
#                  "selisih_vs_A": boot.mean(), "CI95_bawah": np.percentile(boot, 2.5),
#                  "CI95_atas": np.percentile(boot, 97.5), "P(lebih baik)": (boot > 0).mean()})
# print(pd.DataFrame(rows).sort_values("gold_auc", ascending=False).round(4).to_string(index=False))

In [13]:
# import os, itertools, importlib.util, time
# import numpy as np, pandas as pd, torch
# from sklearn.metrics import roc_auc_score
# from tqdm.auto import tqdm

# COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
# WORK = "/kaggle/working"
# LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
#           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
#           "Contusion", "Fracture"]

# # indeks semua file di /kaggle/input SEKALI saja (lewati folder kompetisi & file volume .npz)
# t0 = time.time()
# FILES, VOL_DIRS = {}, {}
# for root, dirs, files in os.walk("/kaggle/input"):
#     dirs[:] = [d for d in dirs if d != "competitions"]
#     for f in files:
#         if f.endswith(".npz"):
#             continue
#         FILES.setdefault(f, os.path.join(root, f))
#     if "_meta.csv" in files:
#         VOL_DIRS["v1"] = root
# print(f"Indeks selesai: {len(FILES)} file dalam {time.time()-t0:.0f} detik")
# print("Folder volume v1:", VOL_DIRS.get("v1"))

# def find_file(name):
#     if name not in FILES:
#         raise FileNotFoundError(name)
#     return FILES[name]

# def mean_auc(y, p):
#     aucs = []
#     for k in range(len(LABELS)):
#         ok = ~np.isnan(y[:, k]) & (y[:, k] != 0.5)
#         yk, pk = y[ok, k], p[ok, k]
#         aucs.append(roc_auc_score(yk, pk) if len(np.unique(yk)) == 2 else np.nan)
#     return float(np.nanmean(aucs))

In [14]:
# out_val, out_gold = f"{WORK}/val_preds_effb0.csv", f"{WORK}/gold_preds_effb0.csv"

# if os.path.exists(out_val):
#     print("Sudah ada, dilewati:", out_val)
# else:
#     spec = importlib.util.spec_from_file_location("tb", find_file("09_train_student.py"))
#     tb = importlib.util.module_from_spec(spec); spec.loader.exec_module(tb)
#     torch.set_num_threads(os.cpu_count())

#     # studi validasi PERSIS sama dengan model lain (split 90/10, random_state=42)
#     val_ids = pd.read_csv(find_file("val_preds_effb0_s2.csv"))["StudyInstanceUID"].tolist()
#     print("Studi val:", len(val_ids), "| CPU thread:", torch.get_num_threads())

#     net = tb.KneeNet("efficientnet_b0", n_views=3, pretrained=False)
#     net.load_state_dict(torch.load(find_file("efficientnet_b0_best.pt"), map_location="cpu"))
#     net = net.eval().to(memory_format=torch.channels_last)

#     ds = tb.KneeDS(val_ids, np.full((len(val_ids), len(LABELS)), np.nan, np.float32),
#                    VOL_DIRS["v1"], train=False)
#     dl = torch.utils.data.DataLoader(ds, batch_size=4, shuffle=False, num_workers=0)

#     ps = []
#     with torch.inference_mode():
#         for vol, vmask, _, _ in tqdm(dl, desc="Prediksi val effb0"):
#             ps.append(torch.sigmoid(net(vol, vmask)).numpy())

#     pv = pd.DataFrame(np.concatenate(ps), columns=LABELS)
#     pv.insert(0, "StudyInstanceUID", val_ids)
#     pv.to_csv(out_val, index=False)
#     pd.read_csv(find_file("gold_preds.csv")).to_csv(out_gold, index=False)
#     print("Selesai:", out_val, "&", out_gold)

In [15]:
# val, gold = {}, {}
# for f, path in FILES.items():
#     if "_kf_f" in f:
#         continue
#     if f.startswith("val_preds_") and f.endswith(".csv"):
#         val[f[len("val_preds_"):-4]] = pd.read_csv(path)
#     if f.startswith("gold_preds_") and f.endswith(".csv"):
#         gold[f[len("gold_preds_"):-4]] = pd.read_csv(path)

# val["effb0"] = pd.read_csv(f"{WORK}/val_preds_effb0.csv")     # baseline seed42 (hasil Cell 1)
# gold["effb0"] = pd.read_csv(f"{WORK}/gold_preds_effb0.csv")

# tags = sorted(set(val) & set(gold))
# print("Model ditemukan:", tags)

# rep = pd.read_csv(find_file("report_labels.csv"))
# tr = pd.read_csv(f"{COMP}/train.csv")
# gold_df = tr[tr[LABELS].notna().all(axis=1)][["StudyInstanceUID"] + LABELS]

# val_ids = val[tags[0]]["StudyInstanceUID"]
# gold_ids = gold[tags[0]]["StudyInstanceUID"]
# for t in tags:
#     assert set(val[t]["StudyInstanceUID"]) == set(val_ids), f"split val {t} berbeda"
#     assert set(gold[t]["StudyInstanceUID"]) == set(gold_ids), f"studi gold {t} berbeda"

# y_val = rep.set_index("StudyInstanceUID").loc[val_ids, LABELS].values.astype(float)
# y_gold = gold_df.set_index("StudyInstanceUID").loc[gold_ids, LABELS].values.astype(float)
# P_val = {t: val[t].set_index("StudyInstanceUID").loc[val_ids, LABELS].values for t in tags}
# P_gold = {t: gold[t].set_index("StudyInstanceUID").loc[gold_ids, LABELS].values for t in tags}
# print(f"Studi val: {len(val_ids)} | studi gold: {len(gold_ids)}")

In [16]:
# print("=== Model tunggal ===")
# single = pd.DataFrame([{"model": t, "val": mean_auc(y_val, P_val[t]),
#                         "gold": mean_auc(y_gold, P_gold[t])} for t in tags])
# print(single.sort_values("val", ascending=False).round(4).to_string(index=False))

# rows = []
# for r in range(2, len(tags) + 1):
#     for combo in itertools.combinations(tags, r):
#         pv = np.mean([P_val[t] for t in combo], axis=0)
#         pg = np.mean([P_gold[t] for t in combo], axis=0)
#         rows.append({"combo": " + ".join(combo), "n": r,
#                      "val": mean_auc(y_val, pv), "gold": mean_auc(y_gold, pg)})
# res = pd.DataFrame(rows)
# res["rata2"] = (res["val"] + res["gold"]) / 2

# print("\n=== 15 kombinasi terbaik menurut VAL ===")
# print(res.sort_values("val", ascending=False).head(15).round(4).to_string(index=False))

# print("\n=== 15 kombinasi terbaik menurut rata-rata VAL & GOLD ===")
# print(res.sort_values("rata2", ascending=False).head(15).round(4).to_string(index=False))

# cands = {
#     "submitted A + student_v2 (LB 0.873)": ["effb0", "effb0_s2", "dinov2s_v2", "effb0_student_v2"],
#     "opsi 1": ["dinov2s_student_v2", "dinov2s_v2", "effb0_student_v2", "effb0_student_v2_s7"],
#     "opsi 2": ["effb0", "effb0_s2", "dinov2s_v2", "effb0_student_v2",
#                "effb0_student_v2_s7", "dinov2s_student_v2"],
# }
# print("\n=== Kandidat submission ===")
# for name, combo in cands.items():
#     pv = np.mean([P_val[t] for t in combo], axis=0)
#     pg = np.mean([P_gold[t] for t in combo], axis=0)
#     print(f"{name:38s} val {mean_auc(y_val, pv):.4f} | gold {mean_auc(y_gold, pg):.4f}")

# res.sort_values("val", ascending=False).to_csv(f"{WORK}/ensemble_search_v2.csv", index=False)

In [17]:
# import os, itertools, importlib.util, time
# import numpy as np, pandas as pd, torch
# from sklearn.metrics import roc_auc_score
# from tqdm.auto import tqdm

# COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
# WORK = "/kaggle/working"
# LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
#           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
#           "Contusion", "Fracture"]

# # indeks semua file di /kaggle/input SEKALI saja (lewati folder kompetisi & file volume .npz)
# t0 = time.time()
# FILES, VOL_DIRS, LABEL_PATHS = {}, {}, []
# for root, dirs, files in os.walk("/kaggle/input"):
#     dirs[:] = [d for d in dirs if d != "competitions"]
#     for f in files:
#         if f.endswith(".npz"):
#             continue
#         FILES.setdefault(f, os.path.join(root, f))
#         if f == "report_labels.csv":
#             LABEL_PATHS.append(os.path.join(root, f))
#     if "_meta.csv" in files:
#         VOL_DIRS["v1"] = root
# print(f"Indeks selesai: {len(FILES)} file dalam {time.time()-t0:.0f} detik")
# print("Folder volume v1:", VOL_DIRS.get("v1"))

# # label acuan val = label gabungan 7B + 32B (dipilih eksplisit kalau ada lebih dari satu file)
# LABEL_PATH = [p for p in LABEL_PATHS if "label-ensemble" in p]
# assert LABEL_PATH, f"label-ensemble belum ditambahkan sebagai input. Ditemukan: {LABEL_PATHS}"
# LABEL_PATH = LABEL_PATH[0]
# print("Label acuan val:", LABEL_PATH)

# def find_file(name):
#     if name not in FILES:
#         raise FileNotFoundError(name)
#     return FILES[name]

# def mean_auc(y, p):
#     """AUC per kelas. Label 0.5 dan kosong diabaikan; label lunak dibinerkan di 0.5."""
#     aucs = []
#     for k in range(len(LABELS)):
#         ok = ~np.isnan(y[:, k]) & (y[:, k] != 0.5)
#         yk, pk = (y[ok, k] > 0.5).astype(float), p[ok, k]
#         aucs.append(roc_auc_score(yk, pk) if len(np.unique(yk)) == 2 else np.nan)
#     return float(np.nanmean(aucs))

In [18]:
# out_val, out_gold = f"{WORK}/val_preds_effb0.csv", f"{WORK}/gold_preds_effb0.csv"

# if os.path.exists(out_val):
#     print("Sudah ada, dilewati:", out_val)
# else:
#     spec = importlib.util.spec_from_file_location("tb", find_file("09_train_student.py"))
#     tb = importlib.util.module_from_spec(spec); spec.loader.exec_module(tb)
#     torch.set_num_threads(os.cpu_count())

#     # studi validasi PERSIS sama dengan model lain (split 90/10, random_state=42)
#     val_ids = pd.read_csv(find_file("val_preds_effb0_s2.csv"))["StudyInstanceUID"].tolist()
#     print("Studi val:", len(val_ids), "| CPU thread:", torch.get_num_threads())

#     net = tb.KneeNet("efficientnet_b0", n_views=3, pretrained=False)
#     net.load_state_dict(torch.load(find_file("efficientnet_b0_best.pt"), map_location="cpu"))
#     net = net.eval().to(memory_format=torch.channels_last)

#     ds = tb.KneeDS(val_ids, np.full((len(val_ids), len(LABELS)), np.nan, np.float32),
#                    VOL_DIRS["v1"], train=False)
#     dl = torch.utils.data.DataLoader(ds, batch_size=4, shuffle=False, num_workers=0)

#     ps = []
#     with torch.inference_mode():
#         for vol, vmask, _, _ in tqdm(dl, desc="Prediksi val effb0"):
#             ps.append(torch.sigmoid(net(vol, vmask)).numpy())

#     pv = pd.DataFrame(np.concatenate(ps), columns=LABELS)
#     pv.insert(0, "StudyInstanceUID", val_ids)
#     pv.to_csv(out_val, index=False)
#     pd.read_csv(find_file("gold_preds.csv")).to_csv(out_gold, index=False)
#     print("Selesai:", out_val, "&", out_gold)

In [19]:
# val, gold = {}, {}
# for f, path in FILES.items():
#     if "_kf" in f:                      # lewati prediksi per fold guru (lama dan baru)
#         continue
#     if f.startswith("val_preds_") and f.endswith(".csv"):
#         val[f[len("val_preds_"):-4]] = pd.read_csv(path)
#     if f.startswith("gold_preds_") and f.endswith(".csv"):
#         gold[f[len("gold_preds_"):-4]] = pd.read_csv(path)

# val["effb0"] = pd.read_csv(f"{WORK}/val_preds_effb0.csv")     # baseline seed42 (hasil Cell 1)
# gold["effb0"] = pd.read_csv(f"{WORK}/gold_preds_effb0.csv")

# tags = sorted(set(val) & set(gold))
# print("Model ditemukan:", tags)

# rep = pd.read_csv(LABEL_PATH)
# tr = pd.read_csv(f"{COMP}/train.csv")
# gold_df = tr[tr[LABELS].notna().all(axis=1)][["StudyInstanceUID"] + LABELS]

# val_ids = val[tags[0]]["StudyInstanceUID"]
# gold_ids = gold[tags[0]]["StudyInstanceUID"]
# for t in tags:
#     assert set(val[t]["StudyInstanceUID"]) == set(val_ids), f"split val {t} berbeda"
#     assert set(gold[t]["StudyInstanceUID"]) == set(gold_ids), f"studi gold {t} berbeda"

# y_val = rep.set_index("StudyInstanceUID").loc[val_ids, LABELS].values.astype(float)
# y_gold = gold_df.set_index("StudyInstanceUID").loc[gold_ids, LABELS].values.astype(float)
# P_val = {t: val[t].set_index("StudyInstanceUID").loc[val_ids, LABELS].values for t in tags}
# P_gold = {t: gold[t].set_index("StudyInstanceUID").loc[gold_ids, LABELS].values for t in tags}
# print(f"Studi val: {len(val_ids)} | studi gold: {len(gold_ids)}")

In [20]:
# print("Label acuan val:", LABEL_PATH)
# print("\n=== Model tunggal (urut gold) ===")
# single = pd.DataFrame([{"model": t, "val": mean_auc(y_val, P_val[t]),
#                         "gold": mean_auc(y_gold, P_gold[t])} for t in tags])
# print(single.sort_values("gold", ascending=False).round(4).to_string(index=False))

# rows = []
# for r in range(2, len(tags) + 1):
#     for combo in itertools.combinations(tags, r):
#         pv = np.mean([P_val[t] for t in combo], axis=0)
#         pg = np.mean([P_gold[t] for t in combo], axis=0)
#         rows.append({"combo": " + ".join(combo), "n": r,
#                      "val": mean_auc(y_val, pv), "gold": mean_auc(y_gold, pg)})
# res = pd.DataFrame(rows)
# res["rata2"] = (res["val"] + res["gold"]) / 2

# print("\n=== 15 kombinasi terbaik menurut rata-rata VAL & GOLD ===")
# print(res.sort_values("rata2", ascending=False).head(15).round(4).to_string(index=False))

# cands = {
#     "LB 0.877 (opsi 3)": ["effb0", "dinov2s_v2", "effb0_student_v2", "effb0_student_v2_s7", "dinov2s_student_v2"],
#     "v3 saja": ["dinov2s_student_v3", "effb0_student_v3", "effb0_student_v3_s7"],
#     "v3 + dinov2 murid v2": ["dinov2s_student_v3", "effb0_student_v3", "effb0_student_v3_s7", "dinov2s_student_v2"],
#     "opsi 3 + v3": ["effb0", "dinov2s_v2", "effb0_student_v2", "effb0_student_v2_s7", "dinov2s_student_v2",
#                     "dinov2s_student_v3", "effb0_student_v3", "effb0_student_v3_s7"],
# }
# print("\n=== Kandidat submission ===")
# for name, combo in cands.items():
#     pv = np.mean([P_val[t] for t in combo], axis=0)
#     pg = np.mean([P_gold[t] for t in combo], axis=0)
#     print(f"{name:24s} val {mean_auc(y_val, pv):.4f} | gold {mean_auc(y_gold, pg):.4f}")

# res.sort_values("rata2", ascending=False).to_csv(f"{WORK}/ensemble_search_v3.csv", index=False)

In [21]:
import os, itertools, importlib.util, time
import numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
WORK = "/kaggle/working"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]

# indeks semua file di /kaggle/input SEKALI saja (lewati folder kompetisi & file volume .npz)
t0 = time.time()
FILES, VOL_DIRS, LABEL_PATHS = {}, {}, []
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = [d for d in dirs if d != "competitions"]
    for f in files:
        if f.endswith(".npz"):
            continue
        FILES.setdefault(f, os.path.join(root, f))
        if f == "report_labels.csv":
            LABEL_PATHS.append(os.path.join(root, f))
    if "_meta.csv" in files:
        VOL_DIRS["v1"] = root
print(f"Indeks selesai: {len(FILES)} file dalam {time.time()-t0:.0f} detik")
print("Folder volume v1:", VOL_DIRS.get("v1"))

# label acuan val = label gabungan 7B + 32B (dipilih eksplisit kalau ada lebih dari satu file)
LABEL_PATH = [p for p in LABEL_PATHS if "label-ensemble" in p]
assert LABEL_PATH, f"label-ensemble belum ditambahkan sebagai input. Ditemukan: {LABEL_PATHS}"
LABEL_PATH = LABEL_PATH[0]
print("Label acuan val:", LABEL_PATH)

def find_file(name):
    if name not in FILES:
        raise FileNotFoundError(name)
    return FILES[name]

def mean_auc(y, p):
    """AUC per kelas. Label 0.5 dan kosong diabaikan; label lunak dibinerkan di 0.5."""
    aucs = []
    for k in range(len(LABELS)):
        ok = ~np.isnan(y[:, k]) & (y[:, k] != 0.5)
        yk, pk = (y[ok, k] > 0.5).astype(float), p[ok, k]
        aucs.append(roc_auc_score(yk, pk) if len(np.unique(yk)) == 2 else np.nan)
    return float(np.nanmean(aucs))

Indeks selesai: 134 file dalam 12 detik
Folder volume v1: /kaggle/input/notebooks/kellenvalerie/rsna-knee-02-preprocess-mri/vol_train
Label acuan val: /kaggle/input/notebooks/kellenvalerie/rsna-knee-01d-label-ensemble/report_labels.csv


In [22]:
out_val, out_gold = f"{WORK}/val_preds_effb0.csv", f"{WORK}/gold_preds_effb0.csv"

if os.path.exists(out_val):
    print("Sudah ada, dilewati:", out_val)
else:
    spec = importlib.util.spec_from_file_location("tb", find_file("09_train_student.py"))
    tb = importlib.util.module_from_spec(spec); spec.loader.exec_module(tb)
    torch.set_num_threads(os.cpu_count())

    # studi validasi PERSIS sama dengan model lain (split 90/10, random_state=42)
    val_ids = pd.read_csv(find_file("val_preds_effb0_s2.csv"))["StudyInstanceUID"].tolist()
    print("Studi val:", len(val_ids), "| CPU thread:", torch.get_num_threads())

    net = tb.KneeNet("efficientnet_b0", n_views=3, pretrained=False)
    net.load_state_dict(torch.load(find_file("efficientnet_b0_best.pt"), map_location="cpu"))
    net = net.eval().to(memory_format=torch.channels_last)

    ds = tb.KneeDS(val_ids, np.full((len(val_ids), len(LABELS)), np.nan, np.float32),
                   VOL_DIRS["v1"], train=False)
    dl = torch.utils.data.DataLoader(ds, batch_size=4, shuffle=False, num_workers=0)

    ps = []
    with torch.inference_mode():
        for vol, vmask, _, _ in tqdm(dl, desc="Prediksi val effb0"):
            ps.append(torch.sigmoid(net(vol, vmask)).numpy())

    pv = pd.DataFrame(np.concatenate(ps), columns=LABELS)
    pv.insert(0, "StudyInstanceUID", val_ids)
    pv.to_csv(out_val, index=False)
    pd.read_csv(find_file("gold_preds.csv")).to_csv(out_gold, index=False)
    print("Selesai:", out_val, "&", out_gold)

Studi val: 435 | CPU thread: 4


Prediksi val effb0:   0%|          | 0/109 [00:00<?, ?it/s]

Selesai: /kaggle/working/val_preds_effb0.csv & /kaggle/working/gold_preds_effb0.csv


In [23]:
val, gold = {}, {}
for f, path in FILES.items():
    if "_kf" in f:                      # lewati prediksi per fold guru (lama dan baru)
        continue
    if f.startswith("val_preds_") and f.endswith(".csv"):
        val[f[len("val_preds_"):-4]] = pd.read_csv(path)
    if f.startswith("gold_preds_") and f.endswith(".csv"):
        gold[f[len("gold_preds_"):-4]] = pd.read_csv(path)

val["effb0"] = pd.read_csv(f"{WORK}/val_preds_effb0.csv")     # baseline seed42 (hasil Cell 1)
gold["effb0"] = pd.read_csv(f"{WORK}/gold_preds_effb0.csv")

tags = sorted(set(val) & set(gold))
print("Model ditemukan:", tags)

rep = pd.read_csv(LABEL_PATH)
tr = pd.read_csv(f"{COMP}/train.csv")
gold_df = tr[tr[LABELS].notna().all(axis=1)][["StudyInstanceUID"] + LABELS]

val_ids = val[tags[0]]["StudyInstanceUID"]
gold_ids = gold[tags[0]]["StudyInstanceUID"]
for t in tags:
    assert set(val[t]["StudyInstanceUID"]) == set(val_ids), f"split val {t} berbeda"
    assert set(gold[t]["StudyInstanceUID"]) == set(gold_ids), f"studi gold {t} berbeda"

y_val = rep.set_index("StudyInstanceUID").loc[val_ids, LABELS].values.astype(float)
y_gold = gold_df.set_index("StudyInstanceUID").loc[gold_ids, LABELS].values.astype(float)
P_val = {t: val[t].set_index("StudyInstanceUID").loc[val_ids, LABELS].values for t in tags}
P_gold = {t: gold[t].set_index("StudyInstanceUID").loc[gold_ids, LABELS].values for t in tags}
print(f"Studi val: {len(val_ids)} | studi gold: {len(gold_ids)}")

Model ditemukan: ['dinov2s', 'dinov2s_student_d32', 'dinov2s_student_v2', 'dinov2s_student_v3', 'dinov2s_v2', 'effb0', 'effb0_s2', 'effb0_student_d32', 'effb0_student_d32_s7', 'effb0_student_v1', 'effb0_student_v2', 'effb0_student_v2_s7', 'effb0_student_v3', 'effb0_student_v3_s7', 'resnet50']
Studi val: 435 | studi gold: 58
